In [2]:
!pip install google-cloud-monitoring

     |████████████████████████████████| 348kB 1.3MB/s eta 0:00:01
     |████████████████████████████████| 296kB 25.5MB/s eta 0:00:01
     |████████████████████████████████| 143kB 37.4MB/s eta 0:00:01
     |████████████████████████████████| 194kB 44.1MB/s eta 0:00:01
     |████████████████████████████████| 51kB 10.0MB/s eta 0:00:01
     |████████████████████████████████| 235kB 36.8MB/s eta 0:00:01
     |████████████████████████████████| 71kB 5.5MB/s  eta 0:00:01
     |████████████████████████████████| 5.6MB 31.3MB/s eta 0:00:01
     |████████████████████████████████| 184kB 4.2MB/s eta 0:00:01
     |████████████████████████████████| 143kB 59.8MB/s eta 0:00:01
     |████████████████████████████████| 71kB 10.7MB/s eta 0:00:01
     |████████████████████████████████| 122kB 44.1MB/s eta 0:00:01
     |████████████████████████████████| 163kB 40.1MB/s eta 0:00:01
     |████████████████████████████████| 92kB 13.1MB/s eta 0:00:01
ERROR: refractml 1.0.3 has requirement cloudpickle==1.6.0, but you'l

In [3]:
from google.cloud import monitoring_v3
import time
project_id = "fdc-development-408605"
pod_name = "jy-e557679f-02f1-4af5-9f8a-e61a2a0754a1-sunny10683796ltimindt"
client = monitoring_v3.MetricServiceClient()
project_name = f"projects/{project_id}"

now = time.time()
seconds = int(now)
nanos = int((now - seconds) * 10 ** 9)
interval = monitoring_v3.TimeInterval(
    {
        "end_time": {"seconds": seconds, "nanos": nanos},
        "start_time": {"seconds": (seconds - 3600), "nanos": nanos},
    }
)
aggregation = monitoring_v3.Aggregation(
    {
        "alignment_period": {"seconds": 1200},  # 20 minutes
        "per_series_aligner": monitoring_v3.Aggregation.Aligner.ALIGN_MEAN,
        "cross_series_reducer": monitoring_v3.Aggregation.Reducer.REDUCE_MEAN,
    }
)

filter_str = f'resource.type="k8s_container" AND resource.label."pod_name"="jy-bda56886-573d-421e-a7d6-d17edb9a1e62-sunny10683796ltimindt" AND metric.type="kubernetes.io/container/cpu/limit_utilization"'

# kubernetes.io/container/cpu/limit_utilization

results = client.list_time_series(
    request={
        "name": project_name,
        "filter": filter_str,
        "interval": interval,
        "view": monitoring_v3.ListTimeSeriesRequest.TimeSeriesView.FULL,
        "aggregation": aggregation,
    }
)
for result in results:
    print(result)


metric {
  type: "kubernetes.io/container/cpu/limit_utilization"
}
resource {
  type: "k8s_container"
  labels {
    key: "project_id"
    value: "fdc-development-408605"
  }
}
metric_kind: GAUGE
value_type: DOUBLE
points {
  interval {
    end_time {
      seconds: 1710938061
      nanos: 818635000
    }
    start_time {
      seconds: 1710938061
      nanos: 818635000
    }
  }
  value {
    double_value: 0.0059005373485265383
  }
}



In [5]:
!curl -H "Metadata-Flavor: Google" http://169.254.169.254/computeMetadata/v1/instance/service-accounts/default/token

  

{"access_token":"ya29.c.c0AY_VpZh5_Mx28wZqghS0efu32W-kthC38HC1R1cfNVRAZke67tTxRYiVULM8iOgTZg0_SHzkFwxlPU0sTosIVlug25mZWymA7BK6aLAVCn9tAbdeLl5SX6UiVTtc9cVujZJkZsuznSWGpmZoJesR1rK917oz3ovzwRmM0IW3X3UiBmmqCD2KE5yuwUs_GCb_TbiKg4c6N6TWSAGFjFBeMnlz0xbT1DBaKyucueEwf1Ml6xe5nEzvjlaBYzlep9Z7XlkqOTheiPwaB7YWJl3Z1acPAXJHvy_sJuUIZ8RqOEfkGNRpLDj2JaWgY5-PwaxvEE91KZRhE0-ogrqZhbN4QjqRKpBTm6eUKJWyZ9VMkXDOnHLN84DVIuZiI01Xfq0_E_mKe44lIBOr0xicBeNBxB0uCIq-LAgLScsg-0rmjGUGXbaDgqN3noKL_zxe1SM3Yu0WvrK-cjys7J225vViezBZdioSD_8yiuDY4OwdoNQo_UF-0h4qVnCBv2H35rxEATiW-29a3LTq8FG2BFuBBj7r1TOCTV1hxTzbJyjxrf9cMwjLk0hMtUK3ogCdIDFcQ9npC8FEp59zwY0-XBSCgU3nB-CYt1Cag29-wDWyEp8N628AI13blhs9w9OgRxJ7WStz39XQgploq3j3IkMIIQ1Y61VuUVp1tYiyrSIu4QjxsRWXOgQhkV4it5UmrWkWkgjby1v_p_8g4sJ90mrshr_40UteiqvUYsyo30-pnzqS8cZY2S_tb6hIqgvfrcM7QaOiM296gXiXpcB861VhviqlUSkFxMrSX1tUgXmbaxJoXp8b0RnudSbgxr83wks-q6JFJnXI_lZpvW0_iUsuBXkFJ8mqi3VsegIYSBfmpQZhQ6Fdy-4QRJvsYci8m9xhRfU3vrfz_vIac8twzFZazqa6O-bdVjJf-mcc8W4g-eRMzeYSWvhsrfjoQ917Bg_d2Vkcu01g9Ye5t3

In [7]:
import google.auth
import requests
from datetime import datetime, timedelta
from urllib.parse import quote

# Get credentials
credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
print('ss')
print(credentials)
print('ss')

# Define the project ID
project_id = "fdc-development-408605"

# Define the current time and interval
end_time = datetime.utcnow()
start_time = end_time - timedelta(hours=1)

# Define filter string
filter_str = 'resource.type="k8s_container" AND resource.label."pod_name"="jy-bda56886-573d-421e-a7d6-d17edb9a1e62-sunny10683796ltimindt" AND metric.type="kubernetes.io/container/cpu/limit_utilization"'
encoded_filter_str = quote(filter_str, safe='')

# Construct the API URL
url = f'https://monitoring.googleapis.com/v3/projects/{project_id}/timeSeries?filter={encoded_filter_str}&interval.startTime={start_time.strftime("%Y-%m-%dT%H:%M:%SZ")}&interval.endTime={end_time.strftime("%Y-%m-%dT%H:%M:%SZ")}'

# Get the access token
token = "ya29.c.c0AY_VpZh5_Mx28wZqghS0efu32W-kthC38HC1R1cfNVRAZke67tTxRYiVULM8iOgTZg0_SHzkFwxlPU0sTosIVlug25mZWymA7BK6aLAVCn9tAbdeLl5SX6UiVTtc9cVujZJkZsuznSWGpmZoJesR1rK917oz3ovzwRmM0IW3X3UiBmmqCD2KE5yuwUs_GCb_TbiKg4c6N6TWSAGFjFBeMnlz0xbT1DBaKyucueEwf1Ml6xe5nEzvjlaBYzlep9Z7XlkqOTheiPwaB7YWJl3Z1acPAXJHvy_sJuUIZ8RqOEfkGNRpLDj2JaWgY5-PwaxvEE91KZRhE0-ogrqZhbN4QjqRKpBTm6eUKJWyZ9VMkXDOnHLN84DVIuZiI01Xfq0_E_mKe44lIBOr0xicBeNBxB0uCIq-LAgLScsg-0rmjGUGXbaDgqN3noKL_zxe1SM3Yu0WvrK-cjys7J225vViezBZdioSD_8yiuDY4OwdoNQo_UF-0h4qVnCBv2H35rxEATiW-29a3LTq8FG2BFuBBj7r1TOCTV1hxTzbJyjxrf9cMwjLk0hMtUK3ogCdIDFcQ9npC8FEp59zwY0-XBSCgU3nB-CYt1Cag29-wDWyEp8N628AI13blhs9w9OgRxJ7WStz39XQgploq3j3IkMIIQ1Y61VuUVp1tYiyrSIu4QjxsRWXOgQhkV4it5UmrWkWkgjby1v_p_8g4sJ90mrshr_40UteiqvUYsyo30-pnzqS8cZY2S_tb6hIqgvfrcM7QaOiM296gXiXpcB861VhviqlUSkFxMrSX1tUgXmbaxJoXp8b0RnudSbgxr83wks-q6JFJnXI_lZpvW0_iUsuBXkFJ8mqi3VsegIYSBfmpQZhQ6Fdy-4QRJvsYci8m9xhRfU3vrfz_vIac8twzFZazqa6O-bdVjJf-mcc8W4g-eRMzeYSWvhsrfjoQ917Bg_d2Vkcu01g9Ye5t3oQQYhFiBtyRzVw1Iru68eanRzetY4YjS-hVZb_Z6q"
print('tojken pr')
print(token)
print('asas')
# Make the API call



response = requests.get(
    url,
    headers={"Authorization": f"Bearer {token}"}
)

# Print the response
print(response.json())


ss
ss
tojken pr
ya29.c.c0AY_VpZh5_Mx28wZqghS0efu32W-kthC38HC1R1cfNVRAZke67tTxRYiVULM8iOgTZg0_SHzkFwxlPU0sTosIVlug25mZWymA7BK6aLAVCn9tAbdeLl5SX6UiVTtc9cVujZJkZsuznSWGpmZoJesR1rK917oz3ovzwRmM0IW3X3UiBmmqCD2KE5yuwUs_GCb_TbiKg4c6N6TWSAGFjFBeMnlz0xbT1DBaKyucueEwf1Ml6xe5nEzvjlaBYzlep9Z7XlkqOTheiPwaB7YWJl3Z1acPAXJHvy_sJuUIZ8RqOEfkGNRpLDj2JaWgY5-PwaxvEE91KZRhE0-ogrqZhbN4QjqRKpBTm6eUKJWyZ9VMkXDOnHLN84DVIuZiI01Xfq0_E_mKe44lIBOr0xicBeNBxB0uCIq-LAgLScsg-0rmjGUGXbaDgqN3noKL_zxe1SM3Yu0WvrK-cjys7J225vViezBZdioSD_8yiuDY4OwdoNQo_UF-0h4qVnCBv2H35rxEATiW-29a3LTq8FG2BFuBBj7r1TOCTV1hxTzbJyjxrf9cMwjLk0hMtUK3ogCdIDFcQ9npC8FEp59zwY0-XBSCgU3nB-CYt1Cag29-wDWyEp8N628AI13blhs9w9OgRxJ7WStz39XQgploq3j3IkMIIQ1Y61VuUVp1tYiyrSIu4QjxsRWXOgQhkV4it5UmrWkWkgjby1v_p_8g4sJ90mrshr_40UteiqvUYsyo30-pnzqS8cZY2S_tb6hIqgvfrcM7QaOiM296gXiXpcB861VhviqlUSkFxMrSX1tUgXmbaxJoXp8b0RnudSbgxr83wks-q6JFJnXI_lZpvW0_iUsuBXkFJ8mqi3VsegIYSBfmpQZhQ6Fdy-4QRJvsYci8m9xhRfU3vrfz_vIac8twzFZazqa6O-bdVjJf-mcc8W4g-eRMzeYSWvhsrfjoQ917Bg_d2Vkcu01g9Ye5t3o

In [8]:
from google.auth import compute_engine

credentials = compute_engine.Credentials()
token = credentials.get_access_token().token

AttributeError: 'Credentials' object has no attribute 'get_access_token'